# Train & Regularize an MLP

**Week 6 · Lesson 7** &nbsp;·&nbsp; GBUS 564: Predictive Analytics &nbsp;·&nbsp; Fall 2026

<a target="_blank" href="https://colab.research.google.com/github/avorybryant/umw_gbus564_fall26/blob/main/notebooks/week6/lectures/07_train_regularize_mlp.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

*Adapted from ISLP Ch. 10.7 and the Ch. 10.9 Lab where relevant.*

## Setup

Same boilerplate as every notebook this term — see Week 1 Lesson 8 if this is your
first time seeing it.

In [ ]:
# Environment setup — identical across every notebook in the course.
# Installs the course package (dataset loaders + helpers) from the public
# course repo; Colab already has numpy/pandas/scikit-learn, so it's quick.
%pip install -q matplotlib git+https://github.com/avorybryant/umw_gbus564_fall26.git

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import set_config

set_config(display="diagram")
plt.rcParams["figure.figsize"] = (8, 5)

RANDOM_SEED = 564  # GBUS 564 — reuse this seed everywhere for reproducibility
rng = np.random.default_rng(RANDOM_SEED)

## 1. Business Problem

**Business question:** left to train on its own, does a neural network learn who
churns, or just memorize the customers it was shown — and once regularization
stops the memorizing, does it rank customers by churn risk better than the best model this
dataset has seen so far, Week 5's tuned XGBoost?

**Why it matters:** the retention team calls customers in order of predicted
risk, so a better *ranking* means fewer wasted calls. A network is the most
expensive of these models to build, explain, and maintain; it has to earn that
cost on this exact question, held to the same test set as every prior model.
Those prior models aren't refit here: Week 5 Lesson 8 already scored them on
this exact test split, and its numbers are the bar to beat.
Every piece this week named shows up here as a runnable choice — a fully
connected architecture (Lesson 2), ReLU/sigmoid activations (Lesson 3), binary
cross-entropy (Lesson 4), gradient descent epoch by epoch (Lesson 5) — and the
story is Lesson 6's: watch the network overfit as it trains, then apply the
three regularizers and see what they buy. Week 5 searched XGBoost's settings,
and this notebook does the same for the network: Model Selection picks the
learning rate, weight decay, and dropout on validation data before the
regularized network ever sees the test set.

## 2. Data

*Load, describe, and sanity-check the data before touching a model.*

Same dataset as every prior lesson that has touched churn — **Telco Customer
Churn**, one row per customer, account/contract/service features plus a binary
`Churn` label. `load_churn()` in `gbus564_coursework.utils.course_data` loads
and cleans it the same way every time.

In [ ]:
from gbus564_coursework.utils.course_data import load_churn

df = load_churn()
df.head()

In [ ]:
display(df.describe())

display(df.describe(include="object"))

### Data Prep

**Three slices, not two.** The test split is identical to Week 5 Lesson 8's —
same `test_size`, `random_state`, and `stratify` — so every number below lines
up against that notebook's scoreboard. Then a further 20% of the *training* rows is carved
off as a **validation** slice. It's what makes overfitting *visible*: the
training-loss curve only ever falls, so the validation curve is the one that
shows when the network stops learning and starts memorizing — and it's the
curve early stopping (Lesson 6) watches every epoch. The test set stays
untouched until Section 4, so the final comparison is still an honest one.

**Scaling.** A network trains by gradient descent (Lesson 5), stepping downhill
on raw input scales: `TotalCharges` runs into the thousands while
`SeniorCitizen` is 0/1, and left unscaled the loss surface is stretched along
some axes and squeezed along others. `make_nn_preprocessor()` one-hot encodes
the categoricals **and** standardizes the numeric columns. It's only *defined*
here — Section 3 fits it on training rows only.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

X, y = df.drop(columns="Churn"), df["Churn"]
categorical_cols = X.select_dtypes(exclude="number").columns
numeric_cols = X.select_dtypes(include="number").columns

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)
# Validation slice for early stopping — carved from the training rows only
X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=RANDOM_SEED, stratify=y_train
)
print(f"fit rows: {len(X_fit):,}   validation rows: {len(X_val):,}   test rows: {len(X_test):,}")

def make_nn_preprocessor():
    return ColumnTransformer([
        ("encode", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("scale", StandardScaler(), numeric_cols),
    ])

## 3. Model

*Baseline first, then the network — trained plain, then regularized.*

In [ ]:
from sklearn.dummy import DummyClassifier

baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)

**The network.** [Keras](https://keras.io) on a TensorFlow backend, built from
exactly the pieces Lessons 1–5 named: fully connected hidden layers (Lesson 2),
ReLU inside (Lesson 3's default), and a single sigmoid output — Lesson 1's
logistic regression, generalized. It's compiled with **binary cross-entropy**
(Lesson 4's pairing for a sigmoid output) and **Adam**, a gradient-descent
variant (Lesson 5) that adapts its own step size per weight.

`build_mlp()` takes the knobs as arguments — hidden-layer sizes, an L2
penalty (weight decay, `kernel_regularizer`), a dropout rate, and Adam's
learning rate (Lesson 5's ρ) — so the same function builds the plain network,
every candidate in Model Selection, and the final regularized one. `fit_mlp()` trains one, optionally with early stopping. The
preprocessor is fit on the fit rows only, then applied to all three slices.

In [ ]:
%pip install -q keras
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"  # CPU only: hide any NVIDIA GPU (must run before keras loads)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"   # quiet TensorFlow's startup info/warning/error logs

import tensorflow as tf
tf.config.set_visible_devices([], "GPU")    # also hides Apple-silicon (Metal) GPUs; restart the kernel to re-run

import keras
from keras import layers

nn_preprocessor = make_nn_preprocessor().fit(X_fit)
X_fit_enc = nn_preprocessor.transform(X_fit)
X_val_enc = nn_preprocessor.transform(X_val)
X_test_enc = nn_preprocessor.transform(X_test)


def build_mlp(units=(16, 8), l2=0.0, dropout=0.0, learning_rate=1e-3):
    """Hidden layers of the given sizes; l2 adds weight decay, dropout adds a Dropout layer after each.
    learning_rate is Adam's step size (Lesson 5's ρ); 0.001 is Keras's default."""
    penalty = keras.regularizers.l2(l2) if l2 else None
    model = keras.Sequential([keras.Input(shape=(X_fit_enc.shape[1],))])
    for i, n_units in enumerate(units, start=1):
        model.add(layers.Dense(n_units, activation="relu", kernel_regularizer=penalty, name=f"hidden_L{i}"))
        if dropout:
            model.add(layers.Dropout(dropout))
    model.add(layers.Dense(1, activation="sigmoid", name="output"))
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=learning_rate), loss="binary_crossentropy", metrics=["accuracy"])
    return model


def fit_mlp(units=(16, 8), l2=0.0, dropout=0.0, learning_rate=1e-3, early_stopping=True, epochs=60):
    keras.utils.set_random_seed(RANDOM_SEED)
    model = build_mlp(units, l2, dropout, learning_rate)
    callbacks = []
    if early_stopping:
        callbacks.append(keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=10, restore_best_weights=True))
    history = model.fit(
        X_fit_enc, y_fit, validation_data=(X_val_enc, y_val),
        epochs=epochs, batch_size=32, verbose=0, callbacks=callbacks,
    )
    return model, pd.DataFrame(history.history)

In [ ]:
# The plain network, drawn: 46 encoded inputs → 16 → 8 → 1 churn probability.
keras.utils.plot_model(
    build_mlp(), show_shapes=True, show_layer_activations=True, dpi=80
)

**Step 1 — train it plain.** Two hidden layers of 16 and 8 units, no
regularization, a fixed 60 epochs, Adam's default learning rate (0.001).

**Why 16 and 8?** It's sized to the data: 46 encoded inputs → 16 → 8 → 1 is
897 weights and biases, comfortably fewer than the 4,507 rows it trains on.
Halving each layer is a common rule of thumb — each layer compresses what the
one before found, narrowing toward a single churn score. It's a judgment call,
not a searched value; Model Selection below tunes the training knobs, not the
size. `batch_size=32` is Lesson 5's minibatch:
roughly 4,500 fit rows ÷ 32 ≈ 141 updates per epoch.

In [ ]:
plain_mlp, plain_history = fit_mlp(early_stopping=False)
best_epoch = int(plain_history["val_loss"].idxmin()) + 1
print(f"Validation loss bottomed out at epoch {best_epoch} of 60")

### Model Selection

*Week 5 Lesson 8's discipline, applied to the network: choose settings by
validation AUC on the training rows only. The test set stays untouched until
Evaluate.*

Three knobs, each from this week's lessons:

- **Learning rate** — Adam's step size, Lesson 5's ρ. Too large and the steps
  overshoot; too small and training crawls.
- **L2** — the strength of weight decay (Lesson 6's λ).
- **Dropout** — the share of hidden units switched off on each update
  (Lesson 6's φ).

The grid is 2 × 3 × 3 = **18 settings**, each trained with early stopping
(patience 10, up to 100 epochs) and scored by **AUC on the validation slice** —
the same metric Evaluate reports, for the same reason Week 5 gave: the
retention team uses the *ranking*. A learning rate of 0.0003 was left out
because at that pace most runs were still improving when they hit the epoch
cap.

**Why one validation slice, not Week 5's 5-fold cross-validation?** Cost. One
network takes about 15 seconds to train; 5-fold CV would make this 90 fits
instead of 18. The trade-off is that early stopping and model selection both
use the same validation slice, so the winning validation score is a little
optimistic — which is exactly why the test set, untouched until Evaluate,
gets the final word. Expect this cell to take about 5 minutes.

In [ ]:
import itertools
import time
from sklearn.metrics import roc_auc_score

grid = {
    "learning_rate": [1e-3, 3e-3],
    "l2": [1e-4, 1e-3, 1e-2],
    "dropout": [0.0, 0.2, 0.4],
}
results = []
best_val_auc = 0.0  # keep the best network so far, so Step 2 can reuse it
start = time.time()
for learning_rate, l2, dropout in itertools.product(*grid.values()):
    model, history = fit_mlp(l2=l2, dropout=dropout, learning_rate=learning_rate, epochs=100)
    val_auc = roc_auc_score(y_val, model.predict(X_val_enc, verbose=0).ravel())
    results.append({"learning_rate": learning_rate, "l2": l2, "dropout": dropout,
                    "epochs": len(history), "validation AUC": val_auc})
    if val_auc > best_val_auc:
        best_val_auc, best_model, best_history = val_auc, model, history

search = pd.DataFrame(results).sort_values("validation AUC", ascending=False).reset_index(drop=True)
print(f"{len(search)} settings in {(time.time() - start) / 60:.1f} minutes")

In [ ]:
best = search.iloc[0]
print(f"Chosen: learning rate {best['learning_rate']:g}, L2 {best['l2']:g}, dropout {best['dropout']:g} "
      f"(validation AUC {best['validation AUC']:.3f})")
print(f"All 18 settings span {search['validation AUC'].min():.3f} to {search['validation AUC'].max():.3f}")

# Every setting at a glance: rows = learning rate × L2, columns = dropout
search.pivot_table(index=["learning_rate", "l2"], columns="dropout", values="validation AUC").round(3)

**Reading the search.** The winner is a learning rate of 0.001, L2 of 0.001,
and dropout of 0.4. But look at the spread before reading too much into that:
all 18 settings land between 0.845 and 0.853 validation AUC. On 1,127
validation customers, gaps that small are within noise; a different split or seed could reorder the top of this table.

Two patterns hold across the whole grid, though. The faster learning rate
(0.003) scores the same or worse than 0.001 in every row: its steps are too big
to settle into the best weights. And the heaviest weight decay (L2 = 0.01) never
wins: at every learning rate and dropout, a lighter L2 scores at least as high —
it shrinks weights the data genuinely needs. More
dropout helps in most rows.

The takeaway isn't that 0.4 is the magic dropout rate. It's that the search
confirms the network is close to its ceiling on this data: no setting in the grid
rescues it, which matters when Section 6 decides whether a network is worth it.

**Step 2 — the chosen network.** Same architecture, now with all three of
Lesson 6's tools at the values Model Selection picked: weight decay, dropout
after each hidden layer, and early stopping — stop once validation loss hasn't
improved for 10 epochs (the **patience**), then roll back to the best epoch's
weights. The search already trained it, so there's nothing to fit again: this
pulls the winning network and its training curves out of the search, to sit
next to Step 1's.

In [ ]:
reg_mlp, reg_history = best_model, best_history  # trained during the search
print(f"Early stopping ended training after {len(reg_history)} epochs")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), sharey=True)
for ax, history, title in [
    (axes[0], plain_history, "Plain — runs the full 60 epochs"),
    (axes[1], reg_history, "Regularized — early-stopped"),
]:
    ax.plot(history.index + 1, history["loss"], label="Training loss", color="#1A73E8")
    ax.plot(history.index + 1, history["val_loss"], label="Validation loss", color="#D93025")
    ax.set_title(title)
    ax.set_xlabel("Epoch")
axes[0].set_ylabel("Binary cross-entropy")
axes[0].axvline(best_epoch, color="#5F6368", linestyle="--", linewidth=1)
axes[0].legend()
plt.tight_layout()

**Reading the two curves.** On the left, the plain network's validation loss
bottoms out at epoch 23 (dashed line) and drifts up from there while training
loss keeps falling — Lesson 6's picture, from a real fit. The rise is gentle on
a dataset this size, but the widening gap is the overfitting signature: the
last 37 epochs went into memorizing training rows. On the right, early
stopping ended training at epoch 43 and rolled back to the best epoch (33,
ten epochs of patience earlier).

One thing that looks wrong but isn't: on the right, training loss sits
*above* validation loss. That's dropout. Training loss is measured with 40% of
the hidden units switched off at random, while validation runs the full
network — dropout is training-only (Lesson 6). The gap that signals
overfitting never opens.

## 4. Evaluate

*Every model, scored once against the same held-out test set — a metric only
means something next to what it is beating.*

**AUC** is the headline: how well each model's scores *order* customers by
churn risk, which is what a call list needs. Accuracy at the 0.5 cutoff is
shown alongside it for continuity with Week 5. The two Week 5 rows are copied
from Lesson 8's scoreboard, not recomputed: same test customers, so the
comparison is like for like.

In [ ]:
from sklearn.metrics import accuracy_score, roc_auc_score

def mlp_proba(model, X_enc):
    return model.predict(X_enc, verbose=0).ravel()

test_proba = {
    "MLP — plain (Step 1)": mlp_proba(plain_mlp, X_test_enc),
    "MLP — regularized (Step 2)": mlp_proba(reg_mlp, X_test_enc),
}
scores = pd.DataFrame({
    "test AUC": {name: roc_auc_score(y_test, p) for name, p in test_proba.items()},
    "test accuracy": {name: accuracy_score(y_test, (p > 0.5).astype(int)) for name, p in test_proba.items()},
})
scores.loc["Baseline (majority class)"] = [0.5, accuracy_score(y_test, baseline.predict(X_test))]

# Not refit — Week 5 Lesson 8's test scores on this identical split
week5_scores = pd.DataFrame(
    {"test AUC": [0.834, 0.832], "test accuracy": [0.801, 0.791]},
    index=["XGBoost, tuned (Week 5 Lesson 8)", "Logistic regression (Week 5 Lesson 8)"],
)
scores = pd.concat([scores, week5_scores])
scores.sort_values("test AUC", ascending=False).round(3)

In [ ]:
auc = scores["test AUC"].drop("Baseline (majority class)").sort_values()
colors = ["#1A73E8" if name.startswith("MLP") else "#9AA0A6" for name in auc.index]
auc.plot.barh(color=colors, title="Test AUC by model (higher is better; gray = Week 5's scores)")
plt.xlim(0.80, 0.84)
plt.xlabel("AUC")
plt.tight_layout()

**What regularization bought.** The plain network is the weakest model on the
board (AUC 0.817). Adding Lesson 6's three tools, at the settings Model
Selection chose, lifts it to 0.827 — a full point of AUC — for a few extra lines
of code and no change to the architecture.
Same layers, same data, same optimizer: the only difference is that the
regularized network was stopped before it could memorize.

It still doesn't reach Week 5's leaders: tuned XGBoost (0.834) and logistic
regression (0.832) both rank customers better. One handicap worth naming: the
networks trained on 80% of the training rows — the rest was the validation
slice early stopping needs — while Week 5's models used all of them. It
isn't what holds the network back: refitting the chosen settings on all the
training rows, for the epoch count early stopping found, lands within a
thousandth or two of 0.827.

## 5. Interpret

*What is the network actually keying off of? Business-readable, not just
statistically significant.*

The network hands back no coefficients (Week 3) and no built-in feature
importances (Week 5) — every signal is spread across hidden layers of weights
with no per-feature meaning on its own. **Permutation importance** sidesteps
that: shuffle one feature's values across the test set, measure how much AUC
drops, put it back, repeat for every feature. It works on any model, which is
exactly the trade being made — a slower, model-agnostic explanation in place
of the model's own accounting of itself.

In [ ]:
from sklearn.inspection import permutation_importance


class KerasEstimatorWrapper:
    """Thin adapter so permutation_importance can call the fitted Keras model."""
    def __init__(self, model):
        self.model = model
    def fit(self, X, y=None):
        return self  # already fit above — sklearn just requires this method to exist
    def predict(self, X):
        return self.model.predict(X, verbose=0).ravel()


def auc_scorer(estimator, X, y_true):
    return roc_auc_score(y_true, estimator.predict(X))


perm = permutation_importance(
    KerasEstimatorWrapper(reg_mlp), X_test_enc, y_test,
    scoring=auc_scorer, n_repeats=5, random_state=RANDOM_SEED,
)
feature_names = [n.replace("encode__", "").replace("scale__", "")
                 for n in nn_preprocessor.get_feature_names_out()]
importance = pd.Series(perm.importances_mean, index=feature_names)
print(f"Shuffling tenure costs {importance['tenure']:.3f} of test AUC")
importance.sort_values().tail(8).plot.barh(
    color="#1A73E8", title="Regularized MLP — permutation importance (test-AUC drop)")
plt.xlabel("Mean AUC drop when shuffled")
plt.tight_layout()

**Tenure dominates.** Shuffling tenure costs the regularized network about 0.085 of
AUC — several times more than any other feature. Contract type
(month-to-month vs. two-year) and monthly charges follow. That's the same story
logistic regression's coefficients and XGBoost's importances told in earlier
weeks: newer, month-to-month, higher-bill customers churn. The network found no
new signal, which fits with it not beating Week 5's models.

Treat this as a first read, not a rigorous one: correlated features (tenure and
`TotalCharges` move together) can trade credit between each other, the problem
SHAP (Week 5 Lessons 10–11) was built to handle more carefully.

## 6. Recommend

**Recommendation:** don't replace Week 5's churn model with a network. Trained
properly — regularized and early-stopped — the network gains a full point of
AUC over its plain version (0.817 → 0.827), but it still ranks customers below
Week 5's tuned XGBoost (0.834) and logistic regression (0.832), and it
costs more to train, explain, and maintain. On a clean tabular dataset of about
5,600 training rows, that extra complexity doesn't earn its keep.

**The rule for when you do use a network:** never ship one trained plain.
Hold out a validation slice, watch its loss, and regularize — early stopping,
weight decay, dropout. Here those three tools moved the network from last
place most of the way to the leaders, without touching the architecture.

**Caveats / next steps:** one random split and one seed — differences of a few
thousandths of AUC are within noise. The learning rate, L2, and
dropout were chosen on a single validation slice, and the search's top settings
sit within noise of each other. The architecture (16 and 8 units) and the patience weren't searched;
they're the next knobs to try. Lesson 8 (Neural Networks on Business Tables) explains why
tabular data like this tends to favor simpler models, and Lesson 9 tests a
network built specifically for tables.